# Day 25: Performance Benchmarking — Tooling & Profiling
> *Inference Engineering* — Chapter 4.5 | Philip Kiely, Baseten Books 2026

**Layer:** Tooling | **Prerequisite:** Phase 1 (Runtime basics)


## Concept Overview

Benchmarking inference systems requires measuring the right metrics: TTFT (Time to First Token), TPOT (Time Per Output Token), and throughput (tokens/sec). Standard tools: `vllm bench` for end-to-end server benchmarking, `torch.profiler` for kernel-level profiling, `nsys` (Nsight Systems) for timeline profiling. The key benchmarking mistakes: single-threaded benchmarks miss batching effects, warm-up is essential (first requests are slow), and prompt length distributions must match production traffic.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from contextlib import contextmanager

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Core Metrics: TTFT, TPOT, Throughput

Measuring the three key inference latency metrics with a simulated streaming client.


In [ ]:
@contextmanager
def cuda_timer(label=''):
    if torch.cuda.is_available():
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        yield
        end.record()
        torch.cuda.synchronize()
        elapsed = start.elapsed_time(end)
        print(f'{label}: {elapsed:.3f} ms')
    else:
        t0 = time.perf_counter()
        yield
        print(f'{label}: {(time.perf_counter()-t0)*1000:.3f} ms')

def simulate_streaming_inference(prompt_tokens, output_tokens, ttft_ms=50, tpot_ms=20):
    """Simulate a streaming inference call and measure metrics."""
    tokens_received = []
    timestamps = []

    # Prefill (TTFT)
    time.sleep(ttft_ms / 1000)
    first_token_time = time.perf_counter()
    tokens_received.append(1)
    timestamps.append(first_token_time)

    # Decode (TPOT)
    start = time.perf_counter()
    for i in range(output_tokens - 1):
        time.sleep(tpot_ms / 1000)
        tokens_received.append(i + 2)
        timestamps.append(time.perf_counter())

    total_time = timestamps[-1] - (timestamps[0] - ttft_ms/1000)

    return {
        'ttft_ms': ttft_ms,
        'tpot_ms': tpot_ms,
        'total_tokens': output_tokens,
        'total_time_ms': total_time * 1000,
        'throughput_tps': output_tokens / total_time,
    }

# Benchmark different configurations
print('Simulated inference metrics:')
print(f'{"Config":<35} {"TTFT (ms)":>10} {"TPOT (ms)":>10} {"Throughput":>12}')
for (prompt, output, ttft, tpot) in [
    (256, 100, 50, 20),
    (1024, 100, 150, 20),
    (256, 500, 50, 20),
    (256, 100, 50, 5),
]:
    m = simulate_streaming_inference(prompt, output, ttft, tpot)
    label = f'p={prompt} o={output} ttft={ttft}ms tpot={tpot}ms'
    print(f'{label:<35} {m["ttft_ms"]:>10.0f} {m["tpot_ms"]:>10.0f} {m["throughput_tps"]:>11.1f}t/s')


## 2. torch.profiler Usage

PyTorch profiler traces kernel execution, memory allocation, and CPU→GPU synchronization.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def workload_to_profile(device):
    torch.manual_seed(42)
    x = torch.randn(32, 512, dtype=torch.float16, device=device)
    W1 = torch.randn(512, 2048, dtype=torch.float16, device=device)
    W2 = torch.randn(2048, 512, dtype=torch.float16, device=device)
    h = torch.nn.functional.gelu(x @ W1)
    return h @ W2

# Run with profiler
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA] if device == 'cuda'
               else [torch.profiler.ProfilerActivity.CPU],
    record_shapes=True,
    with_flops=True,
) as prof:
    for _ in range(10):
        _ = workload_to_profile(device)

print('Top CUDA kernels by self time:')
print(prof.key_averages().table(sort_by='cuda_time_total' if device == 'cuda' else 'cpu_time_total',
                                 row_limit=8))


## 3. Throughput vs Latency Tradeoff

Batching improves throughput but increases latency. Finding the Pareto-optimal operating point.


In [ ]:
def benchmark_batch_sweep(device, d_model=4096, d_ff=14336, max_batch=64):
    """Measure throughput and latency vs batch size."""
    results = []
    W = torch.randn(d_ff, d_model, device=device, dtype=torch.float16)
    W2 = torch.randn(d_model, d_ff, device=device, dtype=torch.float16)

    for batch in [1, 2, 4, 8, 16, 32, 64]:
        x = torch.randn(batch, d_model, device=device, dtype=torch.float16)
        # Warmup
        for _ in range(5):
            torch.nn.functional.gelu(x @ W.T) @ W2.T
        if device == 'cuda': torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(50):
            h = torch.nn.functional.gelu(x @ W.T)
            out = h @ W2.T
        if device == 'cuda': torch.cuda.synchronize()
        t1 = time.perf_counter()

        latency_ms = (t1 - t0) / 50 * 1000
        throughput = batch / latency_ms * 1000  # tokens/sec
        results.append({'batch': batch, 'latency_ms': latency_ms, 'throughput_tps': throughput})

    return results

device = 'cuda' if torch.cuda.is_available() else 'cpu'
results = benchmark_batch_sweep(device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
batches = [r['batch'] for r in results]
lats = [r['latency_ms'] for r in results]
thrpts = [r['throughput_tps'] for r in results]

ax1.semilogx(batches, lats, 'b-o')
ax1.set_xlabel('Batch Size'); ax1.set_ylabel('Latency (ms)')
ax1.set_title('Latency vs Batch Size')
ax1.grid(True)

ax2.semilogx(batches, thrpts, 'r-o')
ax2.set_xlabel('Batch Size'); ax2.set_ylabel('Throughput (tokens/sec)')
ax2.set_title('Throughput vs Batch Size')
ax2.grid(True)

plt.tight_layout()
plt.savefig('throughput_latency_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'{"Batch":>8} {"Latency (ms)":>15} {"Throughput (t/s)":>18}')
for r in results:
    print(f'{r["batch"]:>8} {r["latency_ms"]:>15.3f} {r["throughput_tps"]:>18.1f}')


## Experiments: Try These

1. **vllm bench**: Run `python -m vllm.entrypoints.openai.api_server` and use `python benchmarks/benchmark_serving.py` with varying concurrency. Plot TTFT and throughput.
2. **Profiler visualization**: Export profiler traces to Chrome trace format (`prof.export_chrome_trace`) and view in `chrome://tracing`.
3. **Warm-up effect**: Benchmark a model with and without warmup requests. How many requests does it take to reach steady-state performance?


## Key Takeaways

- TTFT (prefill latency) and TPOT (decode latency per token) are the two fundamental latency metrics for LLM inference — they have different sensitivity to batch size.
- torch.profiler reveals kernel-level timing: use it to identify whether compute, memory bandwidth, or launch overhead is the bottleneck.
- Throughput and latency are in fundamental tension: larger batches improve throughput by amortizing weight loads but increase per-request latency.
- Always warm up before benchmarking — the first N requests trigger JIT compilation, cache misses, and other one-time costs that distort measurements.

## References
- *Inference Engineering* Ch 4.5 — Philip Kiely, Baseten Books 2026
- vLLM benchmarking documentation
- PyTorch Profiler documentation
